## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

In [3]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator
from agentic.workflow import orchestrator

## Routing demo

Run these three synthetic tickets to compare routing by content and source priority. They use the same A/B variant. The demo runs the actual LangGraph workflow with fixed agent responses and an isolated store, so it needs no model calls and writes no ticket records.


In [4]:
# v1 fixed for rubric
from types import SimpleNamespace
from unittest.mock import patch

from IPython.display import Markdown, display

from agentic import workflow
from agentic.agents.triage_agent import TriageDecision

quota_question = "What is my monthly subscription quota?"
booking_question = "What experiences are available this weekend?"
demo_cases = [
    (quota_question, "normal", "billing"),
    (quota_question, "critical", "escalation"),
    (booking_question, "normal", "reservation"),
]
sample_intents = {quota_question: "billing", booking_question: "reservation"}


def sample_triage(user_input, channel, memory_context):
    return TriageDecision(
        primary_intent=sample_intents[user_input],
        sentiment="neutral",
        urgency="medium",
        confidence=0.95,
        reasoning="Fixed classification for the routing demo.",
        rewritten_query=user_input,
    )


def resolved_reply(response):
    return SimpleNamespace(
        resolved=True,
        needs_escalation=False,
        escalation_reason=None,
        response=response,
    )


demo_graph = workflow.builder.compile(
    checkpointer=workflow.MemorySaver(),
    store=workflow.InMemoryStore(),
)
rows = [
    "| Ticket | Source priority | Classified intent | Visited node | Outcome |",
    "| --- | --- | --- | --- | --- |",
]

with patch.multiple(
    workflow,
    triage_agent=sample_triage,
    billing_agent=lambda **_: resolved_reply("Your plan includes four experiences."),
    reservation_agent=lambda **_: resolved_reply("Here are the available experiences."),
    escalation_agent=lambda **_: SimpleNamespace(
        customer_response="I have forwarded your request for review."
    ),
):
    for index, (text, priority, expected_node) in enumerate(demo_cases, start=1):
        ticket_id = f"routing-demo-{index}"
        state = workflow.normalize_platform_payload(
            {"text": text, "priority": priority, "ticket_id": ticket_id}
        )
        state["ab_variant"] = "variant_a"
        config = {"configurable": {"thread_id": ticket_id}}
        visited = [
            node
            for update in demo_graph.stream(state, config, stream_mode="updates")
            for node in update
        ]
        selected_node = visited[visited.index("triage_node") + 1]
        final_state = demo_graph.get_state(config).values
        assert selected_node == expected_node, visited

        outcome = "Escalated" if final_state["needs_escalation"] else "Resolved"
        if final_state["needs_escalation"]:
            outcome += f" ({final_state['escalation_reason']})"
        intent = final_state["triage"]["primary_intent"]
        rows.append(
            f"| {text} | {final_state['priority']} | {intent} | {selected_node} | {outcome} |"
        )

display(Markdown("\n".join(rows)))


| Ticket | Source priority | Classified intent | Visited node | Outcome |
| --- | --- | --- | --- | --- |
| What is my monthly subscription quota? | normal | billing | billing | Resolved |
| What is my monthly subscription quota? | critical | billing | escalation | Escalated (Source ticket priority is critical.) |
| What experiences are available this weekend? | normal | reservation | reservation | Resolved |

## End-to-end ticket processing and audit demo

Uses an isolated core database, an in-memory Chroma collection seeded from the support articles, and real CultPass lookups. Fixed model responses and local embeddings make every scenario repeatable offline. The final handoff and audit records are written by the same code used by the chat entry point.


In [5]:
# v1 fixed for rubric
import hashlib
import json
import re
import sqlite3
from contextlib import closing
from importlib import import_module
from math import sqrt
from tempfile import TemporaryDirectory
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch
from uuid import uuid4

import chromadb
from sqlalchemy import create_engine

from agentic import workflow
from agentic.agents.billing_agent import BillingResult
from agentic.agents.triage_agent import TriageDecision
from agentic.ticket_processing import query_ticket_events, submit_ticket
from agentic.tools import kb_tool, ticket_tool
from agentic.tools.access_tool import get_account_access_status
from agentic.tools.billing_tool import get_subscription_details

qa = import_module("agentic.agents.general_qa_agent")
policy_question = "How does prorated billing work when I change plans mid-cycle?"
account_question = "What is my subscription tier?"
unsupported_question = "Does CultPass offer physical gift cards for corporate events?"
failure_question = "Simulate a provider failure"


class DemoEmbeddings:
    def embed_query(self, text):
        vector = [0.0] * 1024
        for word in re.findall(r"\b\w{3,}\b", text.lower()):
            index = int.from_bytes(hashlib.sha256(word.encode()).digest()[:4], "big")
            vector[index % len(vector)] += 1
        norm = sqrt(sum(value * value for value in vector)) or 1
        return [value / norm for value in vector]

    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]


def demo_evidence(messages):
    request = json.loads(messages[-1].content)
    excerpts = []
    if request["question"] == policy_question:
        for article_id, article in request["articles"].items():
            if article["title"].startswith("How Prorated Billing"):
                excerpts.append(qa.ArticleExcerpt(
                    article_id=article_id, excerpt=article["body"].split("\n\n", 1)[0],
                ))
    return qa.KnowledgeEvidence(excerpts=excerpts)


def demo_triage(user_input, channel, memory_context):
    if user_input == failure_question:
        raise OSError("Simulated model provider outage")
    return TriageDecision(
        primary_intent="general_qa" if user_input == unsupported_question else "billing",
        sentiment="neutral", urgency="medium", confidence=0.95,
        reasoning="Controlled classification for the end-to-end demo.",
        rewritten_query=user_input, knowledge_only=user_input != account_question,
    )


def demo_billing(rewritten_query, user_id_or_email):
    details = get_subscription_details.invoke({"user_id_or_email": user_id_or_email})
    assert "error" not in details, details.get("error")
    return BillingResult(
        resolved=True, needs_escalation=False, response=f"Your current tier is {details['tier']}.",
    ).record_tool_call(get_subscription_details.name)


seed_path = Path(workflow.__file__).resolve().parents[1] / "data/external/cultpass_users.jsonl"
with seed_path.open(encoding="utf-8") as seed:
    customer_id = json.loads(next(seed))["id"]
client = chromadb.EphemeralClient()
collection = client.create_collection(f"integration-{uuid4().hex}", embedding_function=None)
model = SimpleNamespace(with_structured_output=lambda _: SimpleNamespace(invoke=demo_evidence))

with TemporaryDirectory() as directory:
    core_path = Path(directory) / "core.sqlite3"
    with (
        closing(sqlite3.connect(ticket_tool.UDAHUB_DB_PATH.as_uri() + "?mode=ro", uri=True)) as original,
        closing(sqlite3.connect(core_path)) as copy,
    ):
        original.backup(copy)
    engine = create_engine(f"sqlite:///{core_path}")
    try:
        with (
            workflow.open_workflow(Path(directory) / "memory.sqlite3") as graph,
            patch.multiple(workflow, triage_agent=demo_triage, billing_agent=demo_billing),
            patch.multiple(kb_tool, _get_collection=lambda: collection, _get_embeddings=DemoEmbeddings, engine=engine),
            patch.object(qa, "ChatOpenAI", lambda **_: model),
            patch.dict(qa.os.environ, {"OPENAI_API_KEY": "offline-demo"}),
        ):
            results = {}
            for label, question, expected in [
                ("policy", policy_question, "resolved"),
                ("account", account_question, "resolved"),
                ("knowledge_gap", unsupported_question, "escalated"),
                ("blank", "   ", "rejected"),
                ("provider_failure", failure_question, "failed"),
            ]:
                outcome = submit_ticket(
                    {"text": question, "user_id_or_email": customer_id, "ab_variant": "variant_a"},
                    graph=graph, core_engine=engine,
                )
                assert outcome.status == expected, (label, outcome)
                results[label] = outcome
                print(f"{label}: {outcome.status} | {outcome.reply}")
                for event in query_ticket_events(core_engine=engine, run_id=outcome.run_id):
                    print(json.dumps(event))

        policy_events = query_ticket_events(core_engine=engine, run_id=results["policy"].run_id, kind="retrieval")
        assert policy_events[0]["status"] == "supported" and policy_events[0]["details"]["source_article_ids"]
        assert len(query_ticket_events(core_engine=engine, tool_name="get_subscription_details")) == 1
        assert [event["ticket_id"] for event in query_ticket_events(
            core_engine=engine, kind="outcome", status="escalated"
        )] == [results["knowledge_gap"].ticket_id]
        gap_events = query_ticket_events(core_engine=engine, run_id=results["knowledge_gap"].run_id, kind="retrieval")
        assert gap_events[0]["details"]["candidates"]
        assert all(
            item["similarity"] < qa.MIN_KNOWLEDGE_SIMILARITY
            for item in gap_events[0]["details"]["candidates"]
        )
        assert "below" in gap_events[0]["details"]["reason"]
        assert not gap_events[0]["details"]["source_article_ids"]
        access = get_account_access_status.invoke({"user_id_or_email": customer_id})
        subscription = get_subscription_details.invoke({"user_id_or_email": customer_id})
        assert "is_blocked" in access and "tier" in subscription
        print("Account access result:", {"is_blocked": access["is_blocked"]})
        print("Subscription result:", {key: subscription[key] for key in ("tier", "status", "monthly_quota")})
        for operation in (get_account_access_status, get_subscription_details):
            invalid = operation.invoke({"user_id_or_email": "missing-review-fixture"})
            assert "error" in invalid
            print(f"{operation.name} missing user:", invalid)
        handoff = query_ticket_events(core_engine=engine, run_id=results["knowledge_gap"].run_id, kind="handoff")[0]
        assert handoff["status"] == "saved"
        with closing(sqlite3.connect(core_path)) as db:
            for label, expected_status in [
                ("policy", "resolved"), ("account", "resolved"), ("knowledge_gap", "escalated"),
                ("provider_failure", "processing_failed"),
            ]:
                ticket_id = results[label].ticket_id
                status, tags = db.execute(
                    "SELECT status, tags FROM ticket_metadata WHERE ticket_id = ?", (ticket_id,)
                ).fetchone()
                assert status == expected_status
                count = db.execute(
                    "SELECT COUNT(*) FROM ticket_messages WHERE ticket_id = ? AND role = 'ai'",
                    (ticket_id,),
                ).fetchone()[0]
                assert count == (0 if label == "provider_failure" else 1)
                if label == "knowledge_gap":
                    assert "escalated:general_support_lead" in tags and "priority:medium" in tags
                    note = db.execute(
                        "SELECT content FROM ticket_messages WHERE ticket_id = ? AND role = 'system'", (ticket_id,)
                    ).fetchone()[0]
                    assert note.startswith(f"[{handoff['details']['escalation_id']}]")
            assert db.execute(
                "SELECT 1 FROM tickets WHERE ticket_id = ?", (results["blank"].ticket_id,)
            ).fetchone() is None
    finally:
        engine.dispose()
        client.delete_collection(collection.name)


policy: resolved | From How Prorated Billing Works When Upgrading or Downgrading:
When a user switches between Basic and Premium mid-cycle, CultPass charges or credits the prorated difference for the remaining days in the billing period.
{"timestamp": "2026-11-22T19:55:59Z", "run_id": "a0a1a5fa-cd01-451d-8570-4e4c1682e16c", "ticket_id": "a0a1a5fa-cd01-451d-8570-4e4c1682e16c", "kind": "submission", "node": null, "status": "accepted", "details": {"channel": "chat", "priority": null}}
{"timestamp": "2026-11-22T19:55:59Z", "run_id": "a0a1a5fa-cd01-451d-8570-4e4c1682e16c", "ticket_id": "a0a1a5fa-cd01-451d-8570-4e4c1682e16c", "kind": "classification", "node": "triage_node", "status": "completed", "details": {"primary_intent": "billing", "confidence": 0.95, "sentiment": "neutral", "urgency": "medium", "knowledge_only": true}}
{"timestamp": "2026-11-22T19:55:59Z", "run_id": "a0a1a5fa-cd01-451d-8570-4e4c1682e16c", "ticket_id": "a0a1a5fa-cd01-451d-8570-4e4c1682e16c", "kind": "route", "node": "tr

## Knowledge retrieval and escalation demo

Run after the setup cells with the configured OpenAI credentials and the seeded knowledge base. Each sample runs the real ChromaDB search and General QA evidence check. The demo fixes Triage to the knowledge path so the outcome depends on article evidence. It uses simulated handoff text and an isolated graph store for the sample tickets.


In [6]:
# v1 fixed for rubric
import os
from importlib import import_module
from types import SimpleNamespace
from unittest.mock import patch

from IPython.display import Markdown, display

from agentic import workflow
from agentic.agents.triage_agent import TriageDecision

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Configure OPENAI_API_KEY and rerun the setup cells before this demo.")

qa_module = import_module("agentic.agents.general_qa_agent")
search_tool = qa_module.semantic_kb_search
knowledge_cases = [
    (
        "How does prorated billing work when I change plans mid-cycle?",
        "billing",
        "How Prorated Billing Works When Upgrading or Downgrading",
    ),
    (
        "What should I do if my QR code will not scan at the venue?",
        "reservation",
        "QR Code Not Scanning or Missing at Venue",
    ),
    ("Does CultPass offer physical gift cards for corporate events?", "general_qa", None),
]
knowledge_intents = {question: intent for question, intent, _ in knowledge_cases}
retrievals = []


def knowledge_demo_triage(user_input, channel, memory_context):
    return TriageDecision(
        primary_intent=knowledge_intents[user_input],
        sentiment="neutral",
        urgency="low",
        confidence=0.95,
        reasoning="Controlled triage to exercise the knowledge path.",
        rewritten_query=user_input,
        knowledge_only=True,
    )


def record_knowledge_search(payload):
    matches = search_tool.invoke(payload)
    retrievals.append(matches)
    return matches


knowledge_graph = workflow.builder.compile(
    checkpointer=workflow.MemorySaver(),
    store=workflow.InMemoryStore(),
)
verification_failures = []
with patch.object(qa_module, "semantic_kb_search", SimpleNamespace(invoke=record_knowledge_search)), patch.multiple(
    workflow,
    triage_agent=knowledge_demo_triage,
    escalation_agent=lambda **_: SimpleNamespace(
        customer_response="A support specialist can review your question."
    ),
):
    for index, (question, _, expected_title) in enumerate(knowledge_cases, start=1):
        retrievals.clear()
        ticket_id = f"knowledge-demo-{index}"
        state = workflow.normalize_platform_payload({"text": question, "ticket_id": ticket_id})
        state["ab_variant"] = "variant_a"
        config = {"configurable": {"thread_id": ticket_id}}
        visited = [
            node
            for update in knowledge_graph.stream(state, config, stream_mode="updates")
            for node in update
        ]
        result = knowledge_graph.get_state(config).values
        matches = [match for batch in retrievals for match in batch if match.get("article_id")]
        source_ids = result["source_article_ids"]
        source_titles = {match["title"] for match in matches if match["article_id"] in source_ids}
        lines = [
            f"### {question}",
            "",
            f"**Graph path:** {' ? '.join(visited)}",
            "",
            "**Retrieved articles**",
            "",
        ]
        lines.extend(
            f"- {match['title']} (cosine similarity {match['similarity']:.4f}, distance {match['distance']})" for match in matches
        )
        if not matches:
            lines.append("- No articles returned.")
        lines.extend(["", f"**Verified article IDs:** {', '.join(source_ids) or 'None'}", ""])
        if result["needs_escalation"]:
            lines.append(f"**Escalated:** {result['escalation_reason']}")
        elif result["resolved"]:
            lines.append("**Resolved with a verified article excerpt.**")
        lines.extend(["", "**Customer reply**", "", result["messages"][-1].content])
        display(Markdown("\n".join(lines)))

        if visited[visited.index("triage_node") + 1] != "general_qa":
            verification_failures.append(f"Case {index} did not run General QA.")
        elif expected_title:
            if not result["resolved"] or result["needs_escalation"] or expected_title not in source_titles:
                verification_failures.append(f"Case {index} did not resolve with {expected_title}.")
        elif (
            "escalation" not in visited
            or source_ids
            or not matches
            or not (
                result["escalation_reason"] == "The retrieved articles do not answer the question."
                or "similarity cutoff" in result["escalation_reason"]
            )
        ):
            verification_failures.append(f"Case {index} did not escalate for insufficient article evidence.")

assert not verification_failures, "\n".join(verification_failures)


### How does prorated billing work when I change plans mid-cycle?

**Graph path:** ab_assign ? memory_recall ? triage_node ? general_qa ? channel_formatter ? memory_write

**Retrieved articles**

- How Prorated Billing Works When Upgrading or Downgrading (cosine similarity 0.5080, distance 0.9841)
- Monthly Experience Quota and Rollover Rules (cosine similarity 0.2120, distance 1.576)
- How to Cancel or Pause a Subscription (cosine similarity 0.1750, distance 1.65)

**Verified article IDs:** 1b64ecc1-564c-4e2c-9f45-77d493c0ed14

**Resolved with a verified article excerpt.**

**Customer reply**

From How Prorated Billing Works When Upgrading or Downgrading:
When a user switches between Basic and Premium mid-cycle, CultPass charges or credits the prorated difference for the remaining days in the billing period.

### What should I do if my QR code will not scan at the venue?

**Graph path:** ab_assign ? memory_recall ? triage_node ? general_qa ? channel_formatter ? memory_write

**Retrieved articles**

- QR Code Not Scanning or Missing at Venue (cosine similarity 0.4482, distance 1.1036)
- How to Cancel or Pause a Subscription (cosine similarity 0.2236, distance 1.5528)
- How to Cancel an Event Reservation (cosine similarity 0.2236, distance 1.5528)

**Verified article IDs:** 230b8882-afba-48e1-b47c-2865da117c1d

**Resolved with a verified article excerpt.**

**Customer reply**

From QR Code Not Scanning or Missing at Venue:
If a user's QR code is not visible or fails to scan:
- Ask them to refresh the 'My Reservations' tab and increase screen brightness.
- Confirm they are logged into the same email used for the booking.
- Provide their 6-character `reservation_id` so the venue can check them in manually.

### Does CultPass offer physical gift cards for corporate events?

**Graph path:** ab_assign ? memory_recall ? triage_node ? general_qa ? escalation ? channel_formatter ? memory_write

**Retrieved articles**

- How to Cancel or Pause a Subscription (cosine similarity 0.1581, distance 1.6838)
- What's Included in a CultPass Subscription (cosine similarity 0.1238, distance 1.7524)
- How to Reserve a Spot for an Event (cosine similarity 0.1103, distance 1.7793)

**Verified article IDs:** None

**Escalated:** Knowledge retrieval confidence is below the 0.25 similarity cutoff.

**Customer reply**

A support specialist can review your question.

## Persistent customer memory demo

This demo uses an isolated SQLite database and the real CultPass subscription lookup. Fixed Triage and reply logic make preference recall visible without an LLM call. It runs a follow-up on the same thread, reopens the database, starts a new ticket for the returning customer, and compares that reply with a different customer's reply.


In [7]:
# v1 fixed for rubric
import json
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory
from types import SimpleNamespace
from unittest.mock import patch

from langchain_core.messages import HumanMessage, ToolMessage

from agentic import workflow
from agentic.agents.billing_agent import BillingResult
from agentic.agents.triage_agent import CustomerPreference, TriageDecision
from agentic.tools.billing_tool import get_subscription_details

seed_path = Path(workflow.__file__).resolve().parents[1] / "data/external/cultpass_users.jsonl"
with seed_path.open(encoding="utf-8") as seed:
    demo_customers = [json.loads(line)["id"] for line in seed if line.strip()]
returning_customer, other_customer = demo_customers[:2]
preference = "Please keep future replies brief."
triage_context = {}


def memory_demo_triage(user_input, channel, memory_context):
    triage_context[user_input] = memory_context
    return TriageDecision(
        primary_intent="billing",
        sentiment="neutral",
        urgency="medium",
        confidence=0.95,
        reasoning="Controlled intent for the memory demo.",
        rewritten_query=user_input,
        customer_preferences=[
            CustomerPreference(category="reply_style", quote=preference)
        ] if preference in user_input else [],
    )


def memory_demo_billing(rewritten_query, user_id_or_email):
    details = get_subscription_details.invoke({"user_id_or_email": user_id_or_email})
    assert "error" not in details, details.get("error")
    tier = details["tier"]
    brief = preference in rewritten_query
    response = f"Tier: {tier}." if brief else f"Your current CultPass subscription tier is {tier}."
    return BillingResult(
        resolved=True,
        needs_escalation=False,
        response=response,
    ).record_tool_messages([
        ToolMessage(
            content="Subscription details retrieved.",
            name=get_subscription_details.name,
            tool_call_id="demo-subscription-lookup",
        )
    ])


def ask_with_memory(graph, thread_id, customer_id, text):
    return graph.invoke(
        {
            "messages": [HumanMessage(content=text)],
            "user_id_or_email": customer_id,
            "ab_variant": "variant_a",
        },
        {"configurable": {"thread_id": thread_id}},
    )


with TemporaryDirectory() as directory, patch.multiple(
    workflow,
    triage_agent=memory_demo_triage,
    billing_agent=memory_demo_billing,
):
    db_path = Path(directory) / "memory-demo.sqlite3"
    first_thread = "memory-first-ticket"
    original_config = {"configurable": {"thread_id": first_thread}}
    with workflow.open_workflow(db_path) as graph:
        ask_with_memory(
            graph, first_thread, returning_customer, f"What is my tier? {preference}"
        )
        follow_up = "Can you repeat that?"
        ask_with_memory(graph, first_thread, returning_customer, follow_up)
        assert "Recent conversation:" in triage_context[follow_up]
        assert "What is my tier?" in triage_context[follow_up]

    with workflow.open_workflow(db_path) as reopened_graph:
        original_state = reopened_graph.get_state(original_config).values
        checkpoints = list(reopened_graph.get_state_history(original_config))
        assert len(original_state["messages"]) == 4 and checkpoints
        assert [call["name"] for call in original_state["tool_usage"]] == [
            "get_subscription_details",
            "get_subscription_details",
        ]

        returning = ask_with_memory(
            reopened_graph, "memory-returning-ticket", returning_customer, "What is my tier now?"
        )
        other = ask_with_memory(
            reopened_graph, "memory-other-ticket", other_customer, "What is my current tier?"
        )
        remembered = json.loads(returning["memory_context"])
        assert remembered["preferences"]["reply_style"] == preference
        assert remembered["resolved_issues"][0]["resolution"]
        assert returning["messages"][-1].content.startswith("Tier:")
        assert other["memory_context"] is None
        assert other["messages"][-1].content.startswith("Your current")

    with closing(sqlite3.connect(db_path)) as database:
        for table in ("checkpoints", "store"):
            columns = [column[1] for column in database.execute(f"PRAGMA table_info({table})")]
            assert columns
            print(f"SQLite {table} columns:", columns)
    print(f"Reopened thread_id={first_thread} with {len(original_state['messages'])} messages and {len(checkpoints)} checkpoints.")
    print("Tool history:", original_state["tool_usage"])
    print("Recalled preferences:", remembered["preferences"])
    print("Recalled resolution:", remembered["resolved_issues"][0]["resolution"])
    print("Returning customer:", returning["messages"][-1].content)
    print("Different customer:", other["messages"][-1].content)


SQLite checkpoints columns: ['thread_id', 'checkpoint_ns', 'checkpoint_id', 'parent_checkpoint_id', 'type', 'checkpoint', 'metadata']
SQLite store columns: ['prefix', 'key', 'value', 'created_at', 'updated_at', 'expires_at', 'ttl_minutes']
Reopened thread_id=memory-first-ticket with 4 messages and 16 checkpoints.
Tool history: [{'name': 'get_subscription_details', 'status': 'success'}, {'name': 'get_subscription_details', 'status': 'success'}]
Recalled preferences: {'reply_style': 'Please keep future replies brief.'}
Recalled resolution: Tier: premium.
Returning customer: Tier: premium.
Different customer: Your current CultPass subscription tier is premium.


In [ ]:
# v1 fixed for rubric
from uuid import uuid4

customer_id = input("CultPass customer ID or email (optional): ").strip() or None
ticket_id = input("Ticket ID to resume (blank for a new ticket): ").strip() or str(uuid4())
print("Ticket ID:", ticket_id)
chat_interface(orchestrator, ticket_id, user_id_or_email=customer_id)


In [ ]:
# v1 fixed for rubric
config = {"configurable": {"thread_id": ticket_id}}
checkpoint = orchestrator.get_state(config)
{
    "messages": checkpoint.values.get("messages", []),
    "tool_usage": checkpoint.values.get("tool_usage", []),
    "checkpoint_count": len(list(orchestrator.get_state_history(config))),
}
